### [주제 : 붓꽃(iris)의 품종 분류] <hr>
- 학습 종류 : 지도학습 - 분류
- 구현 단계
    * (1) 데이터 준비 및 확인
    * (2) 데이터 전처리
    * (3) 학습용|검증용|테스트용 데이터셋 준비
    * (4) 학습 => 교차검증 + 하이퍼파라미터 튜닝
    * (5) 평가

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import koreanlize_matplotlib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.neighbors import KNeighborsClassifier

**[1] 데이터 준비 및 확인**

In [2]:
## 데이터 선정
DATA_FILE = '../Data/Numbers/iris.csv'
dataDF = pd.read_csv(DATA_FILE)

In [3]:
dataDF.info()
display(dataDF.head(3))
display(dataDF.describe(include='all'))

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal.length  150 non-null    float64
 1   sepal.width   150 non-null    float64
 2   petal.length  150 non-null    float64
 3   petal.width   150 non-null    float64
 4   variety       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa


,sepal.length,sepal.width,petal.length,petal.width,variety
count,150.000000,150.000000,150.000000,150.000000,150
unique,NaN,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,Setosa
freq,NaN,NaN,NaN,NaN,50
mean,5.843333,3.057333,3.758000,1.199333,NaN
std,0.828066,0.435866,1.765298,0.762238,NaN
min,4.300000,2.000000,1.000000,0.100000,NaN
25%,5.100000,2.800000,1.600000,0.300000,NaN
50%,5.800000,3.000000,4.350000,1.300000,NaN
75%,6.400000,3.300000,5.100000,1.800000,NaN


In [4]:
# [1차 해석]
# - variety 컬럼 : str => category 형변환
# - 결측치 : 없음
# 

**[2] 데이터 전처리**

In [5]:
# variety 컬럼 : str => category 형변환
dataDF['variety'] = dataDF['variety'].astype('category')
dataDF.dtypes

sepal.length     float64
sepal.width      float64
petal.length     float64
petal.width      float64
variety         category
dtype: object

**[3] 학습용|검증용|테스트용 데이터셋 준비**

In [6]:
## 피처와 타겟 분리
featureDF = dataDF[dataDF.columns[:-1]]
target = dataDF[dataDF.columns[-1]]
print(featureDF.shape, target.shape)

(150, 4) (150,)


In [7]:
# 학습용|테스트용 분리 : 분류 => 타겟 쿨래스 비율 유지 stratify 매개변수 설정
#                     재현성 => random_state 매개변수 설정
#                     피쳐 차원 => 2D, 타겟 차원 => 1D
x_train, x_test, y_train, y_test  = train_test_split(featureDF, target, 
                 test_size=0.2, 
                 random_state=10, 
                 stratify=target)

In [8]:
print(f'{x_train.shape}, {y_train.shape}, {x_test.shape}, {y_test.shape}')
print(f'{y_train.value_counts().tolist()}')
print(f'{y_test.value_counts().tolist()}')

(120, 4), (120,), (30, 4), (30,)
[40, 40, 40]
[10, 10, 10]


**[4] 교차검증 + 튜닝**

In [9]:
## => GridSearchCV 인스턴스 생성 및 진행
## - 모든 조합의 모델 생성
## - 조합된 모델별 교차 검증 진행
## - 최적 조합의 하이퍼파라미터 추출
## - 단점 : 시간이 오래 걸림
## - cv 파라미터
##   * 기본값 : 5
##   * 학습 시 전달하는 y값에 따라서, KFold, StratifiedKFold 설정

# 모델 인스턴스 : KNN
k_model = KNeighborsClassifier()

# => 모델의 하이퍼 파라미터 Dict
param_dict = {'n_neighbors': range(1,51,2), 'p':[1,2], 
              'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']}

# => 교차검증 인스턴스 생성
skFold = StratifiedKFold(shuffle=True, random_state=10)

# => 튜닝 및 교차검증 진행 인스턴스
gs_model = GridSearchCV(k_model, cv=skFold, param_grid=param_dict, return_train_score=True)

In [10]:
# => 하이퍼 파라미터 조합으로 모델 생성 및 교차검증 학습 진행
# => y_train의 값에 따라서 모델 생성 및 교차검증 학습 진행
gs_model.fit(x_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'algorithm': ['auto', 'ball_tree', ...], 'n_neighbors': range(1, 51, 2), 'p': [1, 2]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of 

In [11]:
# => 학습 후 설정된 모델 파라미터 확인
# = cv.results_ : 각 모델의 교차검증 결과 저장 dict 형식
cv_resultDF = pd.DataFrame(gs_model.cv_results_)

# = best_params_ : 최고 성능의 하이퍼파라미터 조합
print(f'gs_model.best_params_ : {gs_model.best_params_} => {gs_model.best_score_:.4f}')

# = best_estimator_ : 최고 성능의 조합으로 재학습한 모델 인스턴스
gs_best_model = gs_model.best_estimator_

gs_model.best_params_ : {'algorithm': 'brute', 'n_neighbors': 9, 'p': 2} => 0.9583


In [12]:
## --------------------------------------------------------------
## => RandomizedSearchCV 인스턴스 생성 및 진행
## - GridSearchCV의 느린 속도 개선
## - 지정된 개수 만큼의 모델을 조합 => 속도 빠름
## - 단점 : GridSearchCV보다 최적의 조합 아닐 수 있음
## - cv 파라미터
##   * 기본값 : 5
##   * 학습 시 전달하는 y값에 따라서, KFold, StratifiedKFold 설정
## - random_state 파라미터
##   * 재현성 위해서 설정
## - n_iter 파라미터
##   * 무작위 조합할 모델 개수 설정
## --------------------------------------------------------------
# 모델 인스턴스 : KNN
k_model = KNeighborsClassifier()

# => 모델의 하이퍼 파라미터 Dict
param_dict = {'n_neighbors': range(1,51,2), 'p':[1,2], 
              'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']}

# => 교차검증 인스턴스 생성
skFold = StratifiedKFold(shuffle=True, random_state=10)

# => 튜닝 및 교차검증 진행 인스턴스
rs_model = RandomizedSearchCV(k_model, param_distributions=param_dict, 
                              cv=skFold, n_iter=50, random_state=10,
                              return_train_score=True, )

In [13]:
# 교차검증 진행
rs_model.fit(x_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'algorithm': ['auto', 'ball_tree', ...], 'n_neighbors': range(1, 51, 2), 'p': [1, 2]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",10
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at 

In [14]:
# 최고 조합으로 재학습된 모델 인스턴스
rs_best_model = rs_model.best_estimator_

# 조합 모델별 학습 성능
cv_resultDF = pd.DataFrame(rs_model.cv_results_)
cv_resultDF.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   mean_fit_time       50 non-null     float64
 1   std_fit_time        50 non-null     float64
 2   mean_score_time     50 non-null     float64
 3   std_score_time      50 non-null     float64
 4   param_p             50 non-null     int64  
 5   param_n_neighbors   50 non-null     int64  
 6   param_algorithm     50 non-null     str    
 7   params              50 non-null     object 
 8   split0_test_score   50 non-null     float64
 9   split1_test_score   50 non-null     float64
 10  split2_test_score   50 non-null     float64
 11  split3_test_score   50 non-null     float64
 12  split4_test_score   50 non-null     float64
 13  mean_test_score     50 non-null     float64
 14  std_test_score      50 non-null     float64
 15  rank_test_score     50 non-null     int32  
 16  split0_train_score  5

**[5] 시각화**

In [15]:
# cv_resultDF.info()

# => 확인하고 싶은 컬럼만 선책
cols = ['rank_test_score', 'mean_test_score', 'mean_train_score']
selDF = cv_resultDF[cols]

In [16]:
selDF = selDF.sort_values(by='rank_test_score')

In [17]:
# # 학습 점수, 검증 점수 그래프
# plt.plot(selDF['K'], selDF[], label = [''])

# # 베스트 K
# plt.vlines(best_k, min_y, max_y, colors='red', linestyles='dotted', label='Best K')
# plt.text(best_k, best_score+0.02, f' <--- Best K = {best_k}', c = 'red')

# # 그래프
# plt.legend()
# plt.xlabel('K')
# plt.ylabel('Score')
# plt.grid()
# plt.show()

**[6] 테스트 데이터 예측**

In [18]:
# # 테스트 데이터 셋으로 예측 및 평가
# k_model = KNeighborsClassifier(n_neighbors=15)
# k_model.fit(x_train, y_train)

# # 예측진행
# y_pre = k_model.predict(x_test)

# # 정답과 예측 진행 : 정답과 오답 개수
# correct = (y_pre == y_test).sum()
# incorrect = y_pre.shape[0] - correct
# print(f'[테스트 예측 평가] 정답 개수 : {correct}개, 오답 개수 : {incorrect}개')

In [22]:
# # 사용자로부터 입력받은 값 예측
# in_data = input("붓꽃 정보 입력(예: 0.12 0.08 1.23 1.44)".split(" "))
# newDF = pd.DataFrame([in_data], columns=x_train.columns)

# y_pre = gs_best_model.predict(newDF)
# print(y_pre)